In [26]:
import pandas as pd
import numpy as np
from scipy import stats
import math

In [140]:
df = pd.read_csv("Salary.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Возраст              9965 non-null   float64
 1   Пол                  10000 non-null  str    
 2   Уровень образования  9022 non-null   str    
 3   Профессия            9481 non-null   str    
 4   Опыт работы          10000 non-null  int64  
 5   Зарплата             10000 non-null  float64
 6   Регион               9535 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 547.0 KB


# Task 1

Test whether the average salary of women is lower than that of men. Significance level 0.01.

In [5]:
df["Пол"].value_counts()

Пол
женский    5023
мужской    4977
Name: count, dtype: int64

In [19]:
salary_men = df[df["Пол"] == "мужской"]["Зарплата"].copy()
salary_women = df[df["Пол"] == "женский"]["Зарплата"].copy() 

In [ ]:
salary_men.info()

In [ ]:
salary_women.info()

The population variances are unknown, so we first test whether they can be assumed to be equal. We use an F-test with a significance level of 0.01.

$H_0$: $\sigma_w^2 = \sigma_m^2$

$H_1$: $\sigma_w^2 \neq \sigma_m^2$

In [55]:
var_women = salary_women.var() 
var_men = salary_men.var() 

F = var_women / var_men
F

np.float64(0.9721240568153208)

In [63]:
var_alpha = 0.01
df_w = salary_women.size - 1
df_m = salary_men.size - 1

f_critical = stats.f.ppf(var_alpha / 2, df_w, df_m), stats.f.ppf(1 - var_alpha / 2, df_w, df_m)

print(f"Critical values for the variance test with {var_alpha} significance level: ({f_critical[0]:.3f}, {f_critical[1]:.3})")
if F < f_critical[0] or F > f_critical[1]:
    print(f"Reject null hypothesis")
else:
    print(f"Fail to reject: we may suppose that the variances are equal")

Critical values for the variance test with 0.01 significance level: (0.930, 1.08)
Fail to reject: we may suppose that the variances are equal


In [56]:
var_p_value = 2 * min(stats.f.sf(F, df_w, df_m), stats.f.cdf(F, df_w, df_m))
var_p_value

np.float64(0.3175773261165909)

The observed value of the test statistic is 1.02 so we do not reject the hypothesis that the variances are equal.

### We now test whether the population mean salaries are equal.

$H_0$: $\mu_w = \mu_m$

$H_1$: $\mu_w < \mu_m$

Since we've decided that the variances are unknown but equal we will use one-tailed test with test statistic

$$
T = \frac{(\bar X_w - \bar X_m) - (\mu_w - \mu_m)}{S_p \sqrt{\frac{1}{n_w} + \frac{1}{n_m}}}
$$
where $S^2_p$ is pooled estimate of variance $\frac{(n_w - 1)S_w^2 + (n_m - 1)S_m^2}{n_w + n_m - 2}$, and T has t-distribution with $n_w + n_m - 2$ degrees of freedom.


In [49]:
mean_alpha = 0.01
t_cricical = stats.t.ppf(mean_alpha, df_w + df_m)
t_cricical

np.float64(-2.32672091328877)

In [ ]:
mean_women = salary_women.mean()
mean_men = salary_men.mean()
var_women = salary_women.var()
var_men = salary_men.var()

std_p = math.sqrt((df_w * var_women + df_m * var_men) / (df_w + df_m))
t = (mean_women - mean_men) / (std_p * math.sqrt(1 / salary_women.size + 1 / salary_men.size))
t

np.float64(0.018315454402022497)

In [ ]:
mean_p_value = stats.t.cdf(t, df_w + df_m)

if t < t_cricical:
    print(f"Since observed value of the test statistic is less than critical value, we reject the null hypothesis. The p-value is {mean_p_value:.3f}")
else:
    print(f"Since the observed test statistic falls to the right of the critical value, we fail to reject the null hypothesis. The p-value is {mean_p_value:.3f}")

Since observed value of the test statistic falled to the right from the critical value, we fail to reject the null hypothesis. The p-value is 0.507


### Use Welch's two-sample t-test

In [62]:
result = stats.ttest_ind(
    salary_women,
    salary_men,
    equal_var=False,
    alternative="less"
)

print(f"t-statistic: {result.statistic:.3f}")
print(f"p-value: {result.pvalue:.4f}")

alpha = 0.01

if result.pvalue < alpha:
    print("Reject H₀: there is a significant difference in mean salaries.")
else:
    print("Fail to reject H₀: there is not enough evidence of a difference in mean salaries.")

t-statistic: 0.018
p-value: 0.5073
Fail to reject H₀: there is not enough evidence of a difference in mean salaries.


# Task 2

Divide the data into two equal-sized samples. Assume that the proportions of people with vocational education are equal in the two samples. Test this assumption at a 2% significance level. (Test this assumption with a 98% confidence level.)

In [68]:
education = df["Уровень образования"].dropna().copy()
education.info()

<class 'pandas.Series'>
Index: 9022 entries, 0 to 9999
Series name: Уровень образования
Non-Null Count  Dtype
--------------  -----
9022 non-null   str  
dtypes: str(1)
memory usage: 141.0 KB


In [102]:
shuffled = education.sample(frac=1, random_state=42)
# shuffled = np.random.permutation(education)
sample_1, sample_2 = (pd.Series(sample, name='education') for sample in np.array_split(shuffled, 2))

$H_0$: $p_1 = p_2$

$H_1$: $p_1 \neq p_2$

Test statistic $Z = \frac{(\hat P_1 - \hat P_2) - (p_1 - p_2)}{\sqrt{\frac{\hat p_1 (1 - \hat p_1)}{n_1} + \frac{\hat p_2 (1 - \hat p_2)}{n_2}}}$ which follows a standard normal distribution under the null hypothesis.

In [72]:
alpha = 0.02
z_critical = (stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2))
z_critical


(np.float64(-2.3263478740408408), np.float64(2.3263478740408408))

In [122]:
p1 = (sample_1 == "среднее профессиональное").mean()
p2 = (sample_2 == "среднее профессиональное").mean()
n1 = sample_1.size
n2 = sample_2.size
print(
    f"Expected counts: "
    f"{n1 * p1:.1f}, {n1 * (1 - p1):.1f}, "
    f"{n2 * p2:.1f}, {n2 * (1 - p2):.1f}"
)
std = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
z_observed = (p1 - p2) / std
print(f"The observed value of the test statistic is {z_observed:.3f}")

p_value = 2 * min(stats.norm.sf(z_observed), stats.norm.cdf(z_observed))

if z_observed < z_critical[0] or z_observed > z_critical[1]:
    print(f"Reject null hypothesi with significance level {alpha}. Value of the test statistic is {z_observed:.3f}, p-value is {p_value:.3f}.")
else:
    print(f"Fail to reject null hypothesis with significance level {alpha}. The critical values are ({z_critical[0]:.3f}, {z_critical[1]:.3f}). Value of the test statistic is {z_observed:.3f}, p-value is {p_value:.3f} ")

Expected counts: 502.0, 4009.0, 493.0, 4018.0
The observed value of the test statistic is 0.302
Fail to reject null hypothesis with significance level 0.02. The critical values are (-2.326, 2.326). Value of the test statistic is 0.302, p-value is 0.762 


### Standard two-proportion z-test.

In [ ]:
x1 = (sample_1 == "среднее профессиональное").sum()
x2 = (sample_2 == "среднее профессиональное").sum()

n1 = sample_1.size
n2 = sample_2.size

p1 = x1 / n1
p2 = x2 / n2
    
p_pooled = (x1 + x2) / (n1 + n2)

std = math.sqrt(
    p_pooled * (1 - p_pooled) * (1 / n1 + 1 / n2)
)

z_observed = (p1 - p2) / std

print(f"The observed value of the test statistic is {z_observed:.3f}")

p_value = 2 * min(stats.norm.sf(z_observed), stats.norm.cdf(z_observed))
if z_observed < z_critical[0] or z_observed > z_critical[1]:
    print(f"Reject null hypothesi with significance level {alpha}. Value of the test statistic is {z_observed:.3f}, p-value is {p_value:.3f}.")
else:
    print(f"Fail to reject null hypothesis with significance level {alpha}. The critical values are ({z_critical[0]:.3f}, {z_critical[1]:.3f}). Value of the test statistic is {z_observed:.3f}, p-value is {p_value:.3f} ")


The observed value of the test statistic is 0.302
Fail to reject null hypothesis with significance level 0.02. The critical values are (-2.326, 2.326). Value of the test statistic is 0.302, p-value is 0.762 


# Task 3

Test the hypothesis about the equality of average salaries for people with less than 5 years of work experience and more than 5 years in the sample at the significance level 0.05.

In [ ]:
labels = df["Опыт работы"] < 5
unexperienced = df[labels]
experienced = df[~labels]

In [128]:
experienced = df.loc[df["Опыт работы"] >= 5, ["Зарплата"]]
inexperienced = df.loc[df["Опыт работы"] < 5, ["Зарплата"]]

$H_0$: $\mu_e = \mu_i$

$H_1$: $\mu_e \neq \mu_i$

We shall perform Independent two-sample t-test

In [129]:
result = stats.ttest_ind(
    experienced,
    inexperienced,
    equal_var=False,
    alternative="two-sided"
)
result

TtestResult(statistic=array([1.49818778]), pvalue=array([0.13430902]), df=array([1405.4951953]))

In [ ]:
alpha = 0.05
n_e = experienced.size
n_i = inexperienced.size

dof = n_e + n_i - 2

t_critical = (stats.t.ppf(alpha / 2, df=df), stats.t.ppf(1 - alpha / 2, df=dof))


In [138]:
t_critical

(np.float64(-1.9602012873568369), np.float64(1.9602012873568369))

# Task 4

Construct a 95% confidence interval for the difference in average salaries between people under 25 and over 55. Test the hypothesis that salaries are equal.

In [141]:
under_25 = df.loc[df["Возраст"] < 25, "Зарплата"]
over_55 = df.loc[df["Возраст"] > 55, "Зарплата"]

In [168]:
result = stats.ttest_ind(
    under_25,
    over_55,
    equal_var=False
)
CI = result.confidence_interval(confidence_level=0.95)

print(f"t-statistic: {result.statistic:.3f}")
print(f"p-value: {result.pvalue:.3f}")
print(f"Welch degrees of freedom: {math.floor(result.df)}")
print(f"95% confidence interval: ({CI[0]:.3f}, {CI[1]:.3f})")


t-statistic: -0.534
p-value: 0.593
Welch degrees of freedom: 1295
95% confidence interval: (-2959.116, 1692.346)


We know that  
$$
\frac{(\bar X_{25} - \bar X_{55}) - (\mu_{25} - \mu_{55})}{\sqrt{\frac{\sigma^2_{25}}{n_{25}} + \frac{\sigma^2_{55}}{n_{55}}}}
$$
which follows a t-distribution with Welch's degrees of freedom.

The Welch's degrees of freedom is equal to 
$$
\frac{(\frac{\sigma^2_{25}}{n_{25}} + \frac{\sigma^2_{55}}{n_{55}})^2}{\frac{{(\frac{\sigma^2_{25}}{n_{25}})^2}}{n_{25} - 1} + \frac{{(\frac{\sigma^2_{55}}{n_{55}})^2}}{n_{55} - 1} }
$$

In [178]:
alpha = 0.05
n_25 = under_25.size
n_55 = over_55.size

x_bar = under_25.mean() - over_55.mean()
se = math.sqrt(under_25.var() / n_25 + over_55.var() / n_55)

var_25 = under_25.var()
var_55 = over_55.var()

df_welch = (
    ( var_25 / n_25 + var_55 / n_55) ** 2
    / 
    (
        (var_25 / n_25) ** 2 / (n_25 - 1)
        + (var_55 / n_55) ** 2 / (n_55 - 1)
    )
)

t_critical = stats.t.ppf(1 - alpha/2, df=df_welch)

margin = t_critical * se

interval = (x_bar - margin, x_bar + margin)

t_statistic = x_bar / se

p_value = 2 * min(stats.t.sf(t_statistic, df=df_welch), stats.t.cdf(t_statistic, df=df_welch))

In [179]:
print(f"t-statistic: {t_statistic:.3f}")
print(f"p-value: {p_value:.3f}")
print(f"Welch degrees of freedom: {math.floor(df_welch)}")
print(f"95% confidence interval: ({interval[0]:.3f}, {interval[1]:.3f})")


t-statistic: -0.534
p-value: 0.593
Welch degrees of freedom: 1295
95% confidence interval: (-2959.116, 1692.346)


# Task 5

Test the hypothesis about the difference in the proportion of people with high salaries (above average) in the sample between Moscow and St. Petersburg. Significance level 0.1.

In [182]:
msk = df.loc[df["Регион"] == "Москва", "Зарплата"]
spb = df.loc[df["Регион"] == "Санкт-Петербург", "Зарплата"]

In [188]:
p_msk = (msk > msk.mean()).mean() # above_avg_fraction
p_spb = (spb > spb.mean()).mean()


Let's perform two-sample z-test for proportions with significance level 0.1.

In [195]:
alpha = 0.1

x_msk = (msk > msk.mean()).sum()
x_spb = (spb > spb.mean()).sum()

n_msk = msk.size
n_spb = spb.size

p_pooled = (x_msk + x_spb) / (n_msk + n_spb)

se = math.sqrt(
    p_pooled * (1 - p_pooled) *
    (1 / n_msk + 1 / n_spb)
)
z_critical = stats.norm.ppf(1 - alpha / 2)

p_statistic = (p_msk - p_spb) / se
p_value = 2 * min(stats.norm.sf(p_statistic), stats.norm.cdf(p_statistic))

if p_statistic < -z_critical or p_statistic > z_critical:
    print(f"Reject null hypothesis about proportion equalities")
else:
    print(f"Fail to reject null hypothesis about the equality of the proportions")

print(f"p-statistic: {p_statistic:.3f}")
print(f"p-value: {p_value:.3f}")
print(f"Critical values: ({-z_critical:.3f}, {z_critical:.3f})")


Fail to reject null hypothesis about the equality of the proportions
p-statistic: 0.840
p-value: 0.401
Critical values: (-1.645, 1.645)
